In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
path = 'D:/SSCode/DST/EDW/'
print

<function print(*args, sep=' ', end='\n', file=None, flush=False)>

In [3]:
orders = pd.read_csv(path + 'olist_orders_dataset.csv')
items = pd.read_csv(path + 'olist_order_items_dataset.csv')
sellers = pd.read_csv(path + 'olist_sellers_dataset.csv')
customers = pd.read_csv(path + 'olist_customers_dataset.csv')
reviews = pd.read_csv(path + 'olist_order_reviews_dataset.csv')
geo = pd.read_csv(path + 'olist_geolocation_dataset.csv')
products = pd.read_csv(path + 'olist_products_dataset.csv')
translation = pd.read_csv(path + 'product_category_name_translation.csv')

In [4]:
# 데이터 병합 (주문-고객-셀러-리뷰-상품)
df = orders.merge(reviews[['order_id', 'review_score']], on='order_id')
df = df.merge(items[['order_id', 'seller_id', 'product_id', 'price']], on='order_id')
df = df.merge(customers[['customer_id', 'customer_zip_code_prefix']], on='customer_id')
df = df.merge(sellers[['seller_id', 'seller_zip_code_prefix']], on='seller_id')
products = products.merge(translation, on='product_category_name', how='left')
df = df.merge(products[['product_id', 'product_category_name_english']], on='product_id', how='left')

In [5]:
# 거리 계산 (Haversine Formula)
# zip code별 평균 좌표로 매핑 (속도 최적화)
geo_avg = geo.groupby('geolocation_zip_code_prefix')[['geolocation_lat', 'geolocation_lng']].mean().reset_index()

In [6]:
# 고객 & 셀러 좌표 결합
df = df.merge(geo_avg, left_on='customer_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='inner').rename(columns={'geolocation_lat': 'cust_lat', 'geolocation_lng': 'cust_lng'})
df = df.merge(geo_avg, left_on='seller_zip_code_prefix', right_on='geolocation_zip_code_prefix', how='inner').rename(columns={'geolocation_lat': 'sell_lat', 'geolocation_lng': 'sell_lng'})

In [7]:
# 거리 계산 함수
def haversine_np(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1)*np.cos(phi2) * np.sin(dlambda/2)**2
    return R * 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

In [8]:
df['dist_km'] = haversine_np(df['cust_lat'], df['cust_lng'], df['sell_lat'], df['sell_lng'])

In [9]:
# 기준: 장거리(800km 이상) vs 단거리, 고만족(4점 이상) vs 저만족
# (브라질 땅덩어리가 커서 800km를 기준으로 잡는 것이 현실적입니다)

conditions = [
    (df['dist_km'] >= 800) & (df['review_score'] >= 4),  # 1. 전국구 확장 (Expansion)
    (df['dist_km'] < 800) & (df['review_score'] <= 2),   # 2. 품질 경보 (Drop)
    (df['dist_km'] >= 800) & (df['review_score'] <= 2),  # 3. 물류 한계 (Geo-Block)
    (df['dist_km'] < 800) & (df['review_score'] >= 4)    # 4. 캐시카우 (Keep)
]
choices = ['💎Expansion (전국확장)', '💣Quality_Fail (판매중단)', '🐢Logistics_Fail (지역차단)', '💰Cash_Cow (유지)']

df['Strategy'] = np.select(conditions, choices, default='Review')

In [11]:
#전략 리포트 출력
summary = df['Strategy'].value_counts(normalize=True) * 100
print("\n📊 [전략 세그먼트 비중]")
print(summary.round(1))


📊 [전략 세그먼트 비중]
Strategy
💰Cash_Cow (유지)            57.4
💎Expansion (전국확장)         18.1
💣Quality_Fail (판매중단)      11.8
Review                     8.4
🐢Logistics_Fail (지역차단)     4.4
Name: proportion, dtype: float64


In [13]:
#[Action Plan] 당장 전국 광고 태워야 할 '히어로 셀러' TOP 10
hero_sellers = df[df['Strategy'] == '💎Expansion (전국확장)'].groupby('seller_id').agg({
    'order_id': 'count',
    'review_score': 'mean',
    'dist_km': 'mean'
}).sort_values('order_id', ascending=False).head(10)

print("\n🏆 [Hero Sellers] 거리가 멀어도 평점이 완벽한 셀러 TOP 10 (즉시 광고 집행):")
display(hero_sellers)


🏆 [Hero Sellers] 거리가 멀어도 평점이 완벽한 셀러 TOP 10 (즉시 광고 집행):


,order_id,review_score,dist_km
seller_id,,,
cc419e0650a3c5ba77189a1882b7556a,356,4.741573,1534.384885
6560211a19b47992c3666cc44a7e94c0,344,4.720930,1624.089081
1f50f920176fa81dab994f9023523100,317,4.776025,1339.403195
06a2c3af7b3aee5d69171b0e14f0ee87,273,4.728938,2114.087457
7a67c85e85bb2ce8582c35f2203ad736,267,4.734082,1421.872434
ea8482cd71df3c1969d7b9473ff13abc,255,4.690196,1489.229032
de722cd6dad950a92b7d4f82673f8833,245,4.665306,1884.558098
53243585a1d6dc2643021fd1853d8905,241,4.626556,1485.532730
955fee9216a65b617aa5c0531780ce60,229,4.615721,1377.171294


In [15]:
#[Action Plan] 당장 퇴출시켜야 할 '불량 카테고리' TOP 5
bad_cats = df[df['Strategy'] == '💣Quality_Fail (판매중단)'].groupby('product_category_name_english').agg({
    'order_id': 'count',
    'review_score': 'mean'
}).sort_values('order_id', ascending=False).head(5)

print("\n🚨 [Blacklist] 가까워도 욕먹는 카테고리 TOP 5 (마케팅 중단):")
display(bad_cats)


🚨 [Blacklist] 가까워도 욕먹는 카테고리 TOP 5 (마케팅 중단):


,order_id,review_score
product_category_name_english,,
bed_bath_table,1692,1.231087
furniture_decor,1236,1.229773
computers_accessories,979,1.186925
sports_leisure,902,1.188470
housewares,859,1.228172
